# 02 · Pré-processamento

Definir separação, transformações e controles contra vazamento. **Sem resultados nesta versão.**

In [19]:
# TODO: transformar decisões aprovadas em funções reutilizáveis de src/.

In [20]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [21]:
# Definição dos Diretórios

raw = Path("../data/raw/Steel_industry_data.csv")

processed = Path("../data/processed")

models = Path("../models")

metadata = Path("../metadata")

### 1) Importação dos Dados

In [22]:
df = pd.read_csv(raw)

df["date"] = pd.to_datetime(df["date"], dayfirst=True)

df = df.sort_values(by="date").reset_index(drop=True)

df.head()

,date,Usage_kWh,Lagging_Current_Reactive.Power_kVarh,Leading_Current_Reactive_Power_kVarh,CO2(tCO2),Lagging_Current_Power_Factor,Leading_Current_Power_Factor,NSM,WeekStatus,Day_of_week,Load_Type
0,2018-01-01 00:00:00,3.42,3.46,0.0,0.0,70.30,100.0,0,Weekday,Monday,Light_Load
1,2018-01-01 00:15:00,3.17,2.95,0.0,0.0,73.21,100.0,900,Weekday,Monday,Light_Load
2,2018-01-01 00:30:00,4.00,4.46,0.0,0.0,66.77,100.0,1800,Weekday,Monday,Light_Load
3,2018-01-01 00:45:00,3.24,3.28,0.0,0.0,70.28,100.0,2700,Weekday,Monday,Light_Load
4,2018-01-01 01:00:00,3.31,3.56,0.0,0.0,68.09,100.0,3600,Weekday,Monday,Light_Load


### 2) Separação dos Dados

In [23]:
target_column = "Load_Type"

y = df[target_column].copy()
X = df.drop(columns=[target_column])

In [24]:
# Split temporal
cut = int(len(df) * 0.8)

X_train, X_test = X.iloc[:cut].copy(), X.iloc[cut:].copy()
y_train, y_test = y.iloc[:cut].copy(), y.iloc[cut:].copy()

print(f'Treino: {X_train['date'].min()} -> {X_train['date'].max()}')
print(f'Teste: {X_test['date'].min()} -> {X_test['date'].max()}')

Treino: 2018-01-01 00:00:00 -> 2018-10-19 23:45:00
Teste: 2018-10-20 00:00:00 -> 2018-12-31 23:45:00


### 3) Seleção das Features

In [25]:
def time_feature_adapt(data: pd.DataFrame) -> pd.DataFrame:
    out = data.copy()
    
    out["hour"] = out["date"].dt.hour + out["date"].dt.minute / 60
    out["month"] = out["date"].dt.month
    out["dayofweek_num"] = out["date"].dt.dayofweek
    
    out['NSM_sin'] = np.sin(2 * np.pi * out['NSM'] / 86400)
    out['NSM_cos'] = np.cos(2 * np.pi * out['NSM'] / 86400)

    return out


X_train_featured = time_feature_adapt(X_train)
X_test_featured = time_feature_adapt(X_test)

C:\Users\rpf97\AppData\Local\Temp\ipykernel_19736\2132601935.py:6: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  out["dayofweek_num"] = out["date"].dt.dayofweek
C:\Users\rpf97\AppData\Local\Temp\ipykernel_19736\2132601935.py:6: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  out["dayofweek_num"] = out["date"].dt.dayofweek


In [26]:
NUMERIC_FEATURES = [
    'Usage_kWh', # feature central
    'Lagging_Current_Reactive.Power_kVarh',  # r=0.9 com o 'Usage_kWh'
    'Leading_Current_Reactive_Power_kVarh',  # r=-0.32 com o 'Usage_kWh'
    'Lagging_Current_Power_Factor', # r=0.39 com o 'Usage_kWh'

    "NSM_sin", "NSM_cos", # substituem o NSM cru, capturam a ciclicidade que o Pearson não prevê
    'hour', 'month', 'dayofweek_num' # adaptação do calendário
]

FEATURE_CLUSTERS_CATEGORICAL = ['Day_of_week']

FEATURE_DISAGARDED = [
    "CO2(tCO2)", # (alta correlação com 'Usage_kwh' -> redundância quade perfeita); mantida em artifacts para enriquerecer o relatório dos clusters
    "Leading_Current_Power_Factor", # variância quase nula, inútil para distância, mas útil para o relatório
    "WeekStatus", # redundante com o Day_of_week
]

DELETED_FEATURE = [
    "date",  # explodida em hour/month/dayofweek_num
    "NSM",   # substituída por NSM_sin / NSM_cos
]

### 4) Pré-Processamento

In [27]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), NUMERIC_FEATURES),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), FEATURE_CLUSTERS_CATEGORICAL)
    ]
)

pipeline = Pipeline(steps=[
    ("prep", preprocessor),
    ("pca",  PCA(n_components=0.90, random_state=42)),
])

X_train_preprocessed = pipeline.fit_transform(X_train_featured)
X_test_preprocessed = pipeline.transform(X_test_featured) # é utilizado somente 'transform' pata evitar vazamento de dados do teste para o treino

components = pipeline.named_steps["pca"].n_components_

print(f"O Pipeline manteve {components} componetes para explicar 90% da variância")

O Pipeline manteve 7 componetes para explicar 90% da variância


In [28]:
# Avaliação das correlações

comp_df = pd.DataFrame(
    X_train_preprocessed,
    columns=[f"PC{i+1}" for i in range(components)]
)
corr_pos = comp_df.corr().abs()

upper = corr_pos.where(np.triu(np.ones(corr_pos.shape), k=1).astype(bool))
print(f"Correlação máxima entre componentes (deve ser ~0): {upper.stack().max().round(6)}")

Correlação máxima entre componentes (deve ser ~0): 0.0


### 5) Salvamento de Dados

In [ ]:
# Processamento de dados
np.save(processed / "x_train.npy", X_train_preprocessed)
np.save(processed / "x_test.npy", X_test_preprocessed)

y_train.to_csv(processed / "y_train.csv", y_train, index=False)
y_test.to_csv(processed / "y_test.csv", y_test, index=False)

TypeError: NDFrame.to_csv() takes from 1 to 2 positional arguments but 3 were given